# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

De façon similaire au TP sur les maillages, il sera demandé de pouvoir retrouver une tangente et un rayon en chaque point, ainsi qu’un point sur la surface définie par le nuage de points.

In [1]:
# Imports
import numpy as np
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 50     # Nombre de points
NUM_NEIGHBORS = 5   # Nombre de voisin par point

### Générer le nuage de points

In [2]:
def generate_regular_spherical_points(radius, num_points):
    # Generating regularly distributed points on the surface of a sphere
    indices = np.arange(0, num_points, dtype=float) + 0.5
    phi = np.arccos(1 - 2*indices/num_points)
    theta = np.pi * (1 + 5**0.5) * indices
    x, y, z = radius * np.cos(theta) * np.sin(phi), radius * np.sin(theta) * np.sin(phi), radius * np.cos(phi)
    points = np.column_stack((x, y, z))
    return points

# Generate regularly distributed points on the surface of a sphere
points = generate_regular_spherical_points(3, NUM_POINTS)

In [3]:
# Create a 3D figure with Plotly
fig = go.Figure()
fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))

# Add points to the figure
fig.add_trace(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='markers',
                           marker=dict(color='magenta', size=3), showlegend=False))

fig.show()

### Obtenir les voisins

In [4]:
def find_direct_neighbors(points, num_neighbors):
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='auto').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = find_direct_neighbors(points, NUM_NEIGHBORS)

In [5]:
# Add connections between neighbors
for i, point in enumerate(points):
    for neighbor_index in direct_neighbors[i]:
        neighbor_point = points[neighbor_index]
        # Add a line trace between the point and its neighbor
        fig.add_trace(go.Scatter3d(x=[point[0], neighbor_point[0]],
                                    y=[point[1], neighbor_point[1]],
                                    z=[point[2], neighbor_point[2]],
                                    mode='lines',
                                    line=dict(color='blue', width=2),
                                    opacity=0.5,
                                    showlegend=False))

fig.show()

### Obtenir les k plus proches voisins

In [6]:
def get_k_near_neighbors(neighbors_indices, k):
    k_near_neighbors = [set() for _ in range(NUM_POINTS)]  # Initialize an array of empty sets
    
    # Add the point itself to its own neighbors
    for point in range(NUM_POINTS):
        k_near_neighbors[point].add(point)
    
    # Iterate through k connections
    for _ in range(k):
        for point in range(NUM_POINTS):
            for neighbor in neighbors_indices[point]:
                k_near_neighbors[point].update(neighbors_indices[neighbor])  # Add neighbors of neighbors
    
    # Remove the point itself from the set of neighbors
    for point in range(NUM_POINTS):
        k_near_neighbors[point].remove(point)
    
    return k_near_neighbors

k_near_neighbors = get_k_near_neighbors(direct_neighbors, 2)

In [7]:
fig.add_trace(go.Scatter3d(x=[points[0, 0]], y=[points[0, 1]], z=[points[0, 2]], mode='markers',
                            marker=dict(color='red', size=5), showlegend=False))

for neighbor_index in k_near_neighbors[0]:
    neighbor_point = points[neighbor_index]
    fig.add_trace(go.Scatter3d(x=[neighbor_point[0]],
                                y=[neighbor_point[1]],
                                z=[neighbor_point[2]],
                                mode='markers',
                                marker=dict(color='orange', size=5),
                                showlegend=False))

fig.show()


### Trouver les rayons

In [8]:
def get_rayons(points, direct_neighbors):
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points, direct_neighbors)

In [9]:
fig = go.Figure()

for point, rayon in zip(points, rayons):
    add_circle(point, rayon, [0,0,0], fig)

fig.update_layout(scene=dict(xaxis=dict(title="X"),
                             yaxis=dict(title="Y"),
                             zaxis=dict(title="Z")),
                  showlegend=False)

# Show the figure
fig.show()